In [ ]:
!pip install category_encoders
!pip uninstall -y tensorflow && pip install tensorflow-cpu
!pip install transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
file_name = '/content/drive/MyDrive/Colab Notebooks/fake_job_postings.csv'
df = pd.read_csv(file_name, encoding='utf-8')
df.head()

#EDA & Visualization

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.isnull().sum()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='fraudulent', data=df)
plt.title('Non-fraudulent(0) vs Fraudulent(1)')
plt.show()

In [ ]:
df['fraudulent'].value_counts(normalize=True) * 100

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x='has_company_logo', hue='fraudulent', data=df, ax=ax1)
ax1.set_title('Có Logo vs Tin giả')

sns.countplot(x='has_questions', hue='fraudulent', data=df, ax=ax2)
ax2.set_title('Có câu hỏi vs Tin giả')

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(x='required_experience', hue='fraudulent', data=df)
plt.title('Mối liên hệ giữa Kinh nghiệm và Tin giả')
plt.xticks(rotation=45)
plt.show()

In [ ]:
df['description_len'] = df['description'].str.len()

plt.figure(figsize=(10, 6))
sns.kdeplot(df[df['fraudulent']==0]['description_len'], label='Real', fill=True)
sns.kdeplot(df[df['fraudulent']==1]['description_len'], label='Fake', fill=True)
plt.title('Phân phối độ dài mô tả công việc')
plt.legend()
plt.show()

In [ ]:
top_fake_industries = df[df['fraudulent']==1]['industry'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top_fake_industries.plot(kind='barh', color='salmon')
plt.title('Top 10 ngành nghề "màu mỡ" cho tin giả')
plt.gca().invert_yaxis()
plt.show()

#Features Preprocessing

In [ ]:
df=df.fillna('')

In [ ]:
df.isnull().sum()

In [ ]:
import torch
import gc
from transformers import DistilBertTokenizer, DistilBertModel
from torch.cuda.amp import autocast

# 1. Thiết lập thiết bị GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang chạy trên: {device}")

# 2. Load DistilBERT
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertModel.from_pretrained(model_name).to(device)

# 3. Kết hợp văn bản
text_features = ['title', 'company_profile', 'description', 'requirements', 'benefits']
df['combined_text'] = df[text_features].agg(' '.join, axis=1)
texts = df['combined_text']

# 4. Tokenize dữ liệu
tokenized_texts = tokenizer(
    texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors='pt',
    add_special_tokens=True
)

# 5. Cấu hình Batch Processing
batch_size = 64
embeddings = []

# 6. Trích xuất Embeddings với tư duy Green AI
model.eval() # Chuyển sang chế độ evaluation
with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        # Lấy dữ liệu từng batch và đưa lên GPU
        input_ids = tokenized_texts["input_ids"][i:i+batch_size].to(device)
        attention_mask = tokenized_texts["attention_mask"][i:i+batch_size].to(device)

        # Sử dụng autocast (FP16) để tiết kiệm 50% bộ nhớ và tăng tốc độ
        with autocast():
            outputs = model(input_ids, attention_mask=attention_mask)

        # Lấy véc-tơ [CLS] (vị trí 0) đại diện cho ngữ nghĩa toàn câu
        # Đưa ngay về CPU để giải phóng bộ nhớ GPU (VRAM)
        batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu()
        embeddings.append(batch_embeddings)

# 7. Tổng hợp và dọn dẹp bộ nhớ
embeddings = torch.cat(embeddings, dim=0)

# Giải phóng bộ nhớ triệt để
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Hoàn thành! Kích thước embeddings: {embeddings.shape}")

In [ ]:
save_path = '/content/drive/MyDrive/Colab Notebooks/distilbert_embeddings.pt'

torch.save(embeddings, save_path)
print(f"Đã lưu embeddings vào: {save_path}")